# CheckAI. Задание 1: атака на линейную модель своими руками

Цель: понять, как устроена логистическая регрессия и почему её можно обмануть малым, почти незаметным изменением входа.

План: данные → логистическая регрессия на numpy → веса как картинка → вывод ∇ₓL → FGSM и график accuracy(ε) → сравнение возмущения с весами.

Перед каждым экспериментом записано **предсказание**, после него — сравнение с результатом.

## 0. Данные

MNIST в CSV (первая колонка — метка, затем 784 пикселя). Оставляем только 3 и 7, пиксели нормируем в [0, 1].
Метка: 7 → 1, 3 → 0.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from numpy.random import RandomState, SeedSequence, MT19937

def load_csv(filepath):
    data = np.loadtxt(filepath, delimiter=',')
    labels, pixels = data[:, 0], data[:, 1:]
    mask = (labels == 3) | (labels == 7)
    x = pixels[mask] / 255.0
    y = (labels[mask] == 7).astype(int).reshape(-1, 1)
    return x, y

x_train, y_train = load_csv("mnist_train/mnist_train.csv")
x_test, y_test = load_csv("mnist_test/mnist_test.csv")
print("train:", x_train.shape, "доля семёрок:", y_train.mean().round(3))
print("test: ", x_test.shape, "доля семёрок:", y_test.mean().round(3))

## 1. Логистическая регрессия на numpy

Модель: $z = w^\top x + b$, $\hat y = \sigma(z) = \frac{1}{1+e^{-z}}$.
Потеря (бинарная кросс-энтропия): $L = -\left[y\ln\hat y + (1-y)\ln(1-\hat y)\right]$, по выборке усредняем.

Градиенты для батча из $m$ объектов: $\nabla_w L = \frac{1}{m} X^\top(\hat y - y)$, $\nabla_b L = \frac{1}{m}\sum(\hat y - y)$.

**Предсказание до эксперимента.** Тройка и семёрка визуально сильно различаются, а линейная модель на MNIST для таких пар
обычно даёт 97–99%. Ожидаю accuracy около 98%. Loss должен быстро упасть с $\ln 2 \approx 0.69$ (при $w\approx 0$) примерно до 0.03–0.05.

In [ ]:
def sigmoid(z):
    # clip нужен, чтобы exp не переполнялся при больших |z|
    return 1 / (1 + np.exp(-np.clip(z, -25, 25)))

def bce(y, p):
    # 1e-15 защищает от log(0)
    return -np.mean(y * np.log(p + 1e-15) + (1 - y) * np.log(1 - p + 1e-15))

def evaluate_accuracy(X, y, w, b):
    return np.mean((sigmoid(X @ w + b) >= 0.5) == y)

rs = RandomState(MT19937(SeedSequence(123456789)))
w = rs.normal(0, 0.01, (784, 1))
b = 0.0
lr, epochs = 1.0, 1000

for epoch in range(epochs):
    p = sigmoid(x_train @ w + b)
    err = p - y_train
    w -= lr * (x_train.T @ err) / len(x_train)
    b -= lr * np.sum(err) / len(x_train)
    if epoch % 200 == 0:
        print(f"Эпоха {epoch}/{epochs} -> Loss: {bce(y_train, p):.4f}")

print(f"\nAccuracy на тесте: {evaluate_accuracy(x_test, y_test, w, b) * 100:.2f}%")

**Результат vs предсказание.** Loss упал с 0.686 до ≈0.03, accuracy на тесте **98.43%** — совпало с ожиданием (≈98%).
Полный градиентный спуск на всей выборке со скоростью `lr=1` сходится устойчиво, потому что задача выпуклая, а пиксели лежат в [0, 1].

## 2. Веса модели как изображение 28×28

**Предсказание.** Вес пикселя показывает, как он влияет на логит класса «7». Ожидаю положительные (красные) области там,
где у семёрки есть штрих, а у тройки нет (верхняя горизонтальная перекладина), и отрицательные (синие) там, где характерна тройка
(нижняя и средняя левые дуги). Края картинки почти всегда нулевые, потому что там нет пикселей (вес не обучается и остаётся около начального значения).

In [ ]:
plt.figure(figsize=(5, 5))
plt.imshow(w.reshape(28, 28), cmap='coolwarm')
plt.title("Веса модели $w$ (красный: за 7, синий: за 3)")
plt.colorbar(label='Значение веса')
plt.axis('off')
plt.show()

**Результат vs предсказание.** Предсказание совпало лишь частично.
- Синие (отрицательные, «за тройку») области — вертикальная полоса в центре и две нижние дуги: именно там у тройки есть штрих, а у семёрки нет. Здесь предсказание верно.
- Красные (положительные, «за семёрку») области — не столько верхняя перекладина, сколько широкий ореол вокруг центра: левая и верхняя части, где у тройки обычно пусто, и диагональ семёрки в центре. Я ожидал более локальной картинки (одна перекладина), а модель «голосует» за семёрку по отсутствию тройки, то есть по пустым участкам. Положительное смещение по краю компенсируется смещением $b$.
- По краям картинки веса остаются около нуля (бледные): пиксели там почти всегда 0, градиент мал, и веса почти не двигаются от начальных ($\mathcal N(0,0.01)$).

## 3. Градиент потерь по входу $x$

Для одного объекта: $z = w^\top x + b$, $\hat y = \sigma(z)$, $L = -y\ln\hat y - (1-y)\ln(1-\hat y)$.

**Шаг 1. Производная по $\hat y$.**
$$\frac{\partial L}{\partial \hat y} = -\frac{y}{\hat y} + \frac{1-y}{1-\hat y} = \frac{\hat y - y}{\hat y(1-\hat y)}$$

**Шаг 2. Производная сигмоиды.** $\sigma'(z) = \sigma(z)(1-\sigma(z)) = \hat y(1-\hat y)$.

**Шаг 3. Цепное правило.**
$$\frac{\partial L}{\partial z} = \frac{\partial L}{\partial \hat y}\cdot\frac{\partial \hat y}{\partial z} = \frac{\hat y - y}{\hat y(1-\hat y)}\cdot \hat y(1-\hat y) = \hat y - y = \sigma(z) - y$$

**Шаг 4. Производная $z$ по входу.** $z = \sum_j w_j x_j + b$, поэтому $\frac{\partial z}{\partial x_j} = w_j$, то есть $\nabla_x z = w$.

**Итог:**
$$\boxed{\nabla_x L = (\sigma(w^\top x + b) - y)\,w}$$

То есть градиент по входу — это вектор весов $w$, умноженный на скаляр $(\hat y - y)$. Направление всегда $\pm w$.
Знак скаляра: $y=1 \Rightarrow \hat y - y < 0$ (градиент $\propto -w$); $y=0 \Rightarrow \hat y - y > 0$ (градиент $\propto +w$).

Проверим формулу численно (центральные конечные разности):

In [ ]:
def loss_single(x, y, w, b):
    p = sigmoid(x @ w + b).item()
    return -(y * np.log(p) + (1 - y) * np.log(1 - p))

i = 0
x0, y0 = x_test[i], y_test[i, 0]
analytic = ((sigmoid(x0 @ w + b) - y0) * w.ravel())

h = 1e-6
numeric = np.zeros(784)
for j in range(784):
    xp, xm = x0.copy(), x0.copy()
    xp[j] += h; xm[j] -= h
    numeric[j] = (loss_single(xp, y0, w, b) - loss_single(xm, y0, w, b)) / (2 * h)

print("макс. расхождение аналитического и численного градиента:", np.abs(analytic - numeric).max())

## 4. Атака FGSM

$x_{adv} = \mathrm{clip}\big(x + \varepsilon\cdot\mathrm{sign}(\nabla_x L),\,0,\,1\big)$.

Идея: каждый пиксель сдвигаем на $\varepsilon$ в сторону, где потеря растёт. Это шаг в норме $\|\cdot\|_\infty \le \varepsilon$.

**Предсказание.** Логит меняется на $w^\top\delta$, а при $\delta=\varepsilon\,\mathrm{sign}(w)$ это $\varepsilon\|w\|_1$: вклады 784 пикселей складываются.
Поэтому даже малое $\varepsilon$ должно сильно ломать модель. Ожидаю: при $\varepsilon=0.05$ падение на несколько процентов, при $0.1$ примерно до 50–70%,
при $0.2$ ниже 15%, при $0.3$ почти 0%. Кривая должна быть похожа на «S» (ступеньку), а не на прямую.

In [ ]:
def fgsm_attack(X, y, w, b, epsilon):
    a = sigmoid(X @ w + b)                       # (m, 1)
    grad_x = (a - y) * w.reshape(1, -1)          # (m, 784): (a - y) * w
    return np.clip(X + epsilon * np.sign(grad_x), 0, 1)

epsilons = [0.0, 0.05, 0.1, 0.15, 0.2, 0.3]
accuracies = []
for eps in epsilons:
    acc = evaluate_accuracy(fgsm_attack(x_test, y_test, w, b, eps), y_test, w, b)
    accuracies.append(acc)
    print(f"eps = {eps:<5} | accuracy под атакой: {acc * 100:.2f}%")

plt.figure(figsize=(6, 4))
plt.plot(epsilons, accuracies, marker='o', color='red')
plt.title(r"Зависимость accuracy от $\varepsilon$")
plt.xlabel(r"$\varepsilon$"); plt.ylabel("Accuracy"); plt.grid(True)
plt.show()

**Результат vs предсказание.** Accuracy: 98.4% → 91.8% ($\varepsilon=0.05$) → 56.2% ($0.1$) → 26.5% ($0.15$) → 9.0% ($0.2$) → 0.05% ($0.3$).
Форма «S» подтвердилась, и предсказание по числам совпало. Модель не просто ошибается, а при $\varepsilon\ge0.2$ ошибается сильнее случайного угадывания (50%):
атака толкает логит именно в сторону неправильного класса. Объекты уже лежат близко к границе, поэтому сдвига логита на несколько единиц достаточно.

Заметьте: $\varepsilon=0.3$ — это сдвиг каждого пикселя на 30% диапазона, на картинке виден шум, но цифра для человека остаётся читаемой.

### Примеры атакованных изображений

**Предсказание.** При $\varepsilon=0.1$ картинки почти не отличаются от оригинала на глаз, но часть предсказаний уже меняется (accuracy 56%).
При $\varepsilon=0.2$ виден зернистый узор, но цифра читается; модель ошибается почти на всех. При $0.3$ шум заметен, цифра всё ещё читается.

In [ ]:
shown = [0.0, 0.1, 0.2, 0.3]
idx = list(range(4))
fig, axes = plt.subplots(len(idx), len(shown), figsize=(2.4 * len(shown), 2.6 * len(idx)))
for r, i in enumerate(idx):
    for c, eps in enumerate(shown):
        xa = fgsm_attack(x_test[i:i+1], y_test[i:i+1], w, b, eps)
        pred = 7 if sigmoid(xa @ w + b).item() >= 0.5 else 3
        true = 7 if y_test[i, 0] == 1 else 3
        axes[r, c].imshow(xa.reshape(28, 28), cmap='gray', vmin=0, vmax=1)
        axes[r, c].set_title(f"eps={eps}\nTrue {true} / Pred {pred}", fontsize=9,
                             color='green' if pred == true else 'red')
        axes[r, c].axis('off')
plt.tight_layout()
plt.show()

**Результат vs предсказание.** Все 4 показанных примера уже при $\varepsilon=0.1$ классифицируются неверно, хотя шум на картинке едва заметен (светло-серый фон и зерно вокруг штриха).
При $\varepsilon=0.3$ шум хорошо виден, но цифры по-прежнему читаются человеком. Предсказание про «читается, но модель ошибается» подтвердилось.
Обратите внимание на тройку (3-я строка): атака одновременно подсвечивает фон там, где $w>0$ (добавляет «ложные штрихи семёрки»), и гасит часть самого штриха тройки там, где $w<0$.

## 5. Сравнение возмущения с весами

Из п. 3: $\mathrm{sign}(\nabla_x L) = \mathrm{sign}(\hat y - y)\cdot\mathrm{sign}(w)$.
Для $y=1$ (семёрка) множитель отрицателен, возмущение $= -\mathrm{sign}(w)$; для $y=0$ (тройка) — $+\mathrm{sign}(w)$.

**Предсказание.** Картинка возмущения должна повторять картинку весов (только в виде $\pm\varepsilon$, без градаций), для семёрок с инвертированными цветами.
Корреляция с $\mathrm{sign}(w)$ — сильная: примерно $+0.8\ldots+1$ для троек и $-0.8\ldots-1$ для семёрок. До $\pm1$ не дойдёт, потому что
`clip` обнуляет изменения там, где пиксель уже 0 и надо уменьшить его дальше (фон), либо уже 1 и надо увеличить.

In [ ]:
eps_cmp = 0.15
X_adv_cmp = fgsm_attack(x_test, y_test, w, b, eps_cmp)
delta = X_adv_cmp - x_test                       # реально добавленное возмущение (после clip)
sign_w = np.sign(w.reshape(-1))

idx3 = np.where(y_test.ravel() == 0)[0][0]
idx7 = np.where(y_test.ravel() == 1)[0][0]

fig, axes = plt.subplots(1, 4, figsize=(14, 3.5))
panels = [
    (w.reshape(28, 28), "Веса $w$"),
    (delta[idx3].reshape(28, 28), "Возмущение, истинный класс 3"),
    (delta[idx7].reshape(28, 28), "Возмущение, истинный класс 7"),
    (delta.mean(axis=0).reshape(28, 28), "Среднее возмущение по тесту"),
]
for ax, (img, title) in zip(axes, panels):
    im = ax.imshow(img, cmap='coolwarm')
    ax.set_title(title, fontsize=9); ax.axis('off')
    fig.colorbar(im, ax=ax, fraction=0.046)
plt.suptitle(rf"Веса и возмущение FGSM ($\varepsilon$ = {eps_cmp})")
plt.show()

for name, i in (("3", idx3), ("7", idx7)):
    print(f"Корреляция возмущения (класс {name}) с sign(w): {np.corrcoef(delta[i], sign_w)[0, 1]:+.3f}")

sign_ay = np.where(y_test.ravel() == 1, -1.0, 1.0).reshape(-1, 1)
expected = sign_ay * sign_w.reshape(1, -1)
changed = delta != 0
match = np.mean(np.sign(delta[changed]) == expected[changed])
print(f"Изменённых пикселей: {changed.mean() * 100:.1f}%, знак совпал с sign(a-y)*sign(w): {match * 100:.1f}%")

dz = ((X_adv_cmp - x_test) @ w).ravel()
print(f"eps*||w||_1 = {eps_cmp * np.abs(w).sum():.2f}, средний |сдвиг логита| = {np.abs(dz).mean():.2f}")

**Результат vs предсказание.** Корреляция $+0.84$ (тройка) и $-0.89$ (семёрка) — как и ожидалось: сильная, со знаком по классу, но не $\pm1$.
У 100% реально изменённых пикселей знак возмущения совпал с $\mathrm{sign}(\hat y - y)\,\mathrm{sign}(w)$. Менялись только 61.3% пикселей: остальные
уже стояли на границе [0, 1] в «неподходящую» сторону, и `clip` их обнулил. Это объясняет, почему корреляция ниже 1.
Средний сдвиг логита 10.4 против верхней границы $\varepsilon\|w\|_1 = 15.97$ — граница недостижима из-за `clip`, но порядок тот же.

**Что видно на картинках.** Возмущение — это «знаковая копия» весов, но разрезанная по `clip`:
- на тёмном фоне пиксель можно только увеличить, поэтому там остаются только красные области возмущения, где нужный знак положителен;
- на самом штрихе пиксель (≈1) можно только уменьшить, поэтому там остаются только синие области;
- серые (нулевые) участки — это места, где нужное направление упирается в границу.

В результате возмущение выглядит как «дорисовать светлое на фоне и притушить сам штрих», а карта возмущения несёт на себе форму самой цифры: среднее возмущение справа явно повторяет контур тройки (синее — там, где обычно штрих).
Узор возмущения при этом задаётся картинкой весов, квантованной до $\pm\varepsilon$: красные области возмущения лежат там, где нужный знак положителен, а синие — где отрицателен. Сравните панели с левой картинкой и проверьте это сами на конкретных пикселях.

**Почему так.** Для линейной модели $z(x+\delta) = z(x) + w^\top\delta$ — сдвиг логита не зависит от $x$ и равен сумме $\sum_j w_j\delta_j$.
При ограничении $\|\delta\|_\infty\le\varepsilon$ эта сумма максимальна, когда каждый $\delta_j$ имеет знак $w_j$ и максимальную величину, то есть $\delta=\varepsilon\,\mathrm{sign}(w)$,
и тогда $w^\top\delta=\varepsilon\|w\|_1$. FGSM для линейной модели — это в точности оптимальная атака в $L_\infty$, а картинка возмущения — «знаковая копия» картинки весов
(с обратным знаком для того класса, который хотим сбросить). По той же причине малый $\varepsilon$ на каждом пикселе даёт большой суммарный эффект: вклады 784 пикселей складываются.

## Итоги

- Логистическая регрессия на 3 vs 7: 98.43% на чистом тесте.
- Модель совершенно не устойчива: при $\varepsilon=0.1$ accuracy 56%, при $\varepsilon=0.3$ — 0.05%.
- Причина — линейность: возмущение $\varepsilon\,\mathrm{sign}(w)$ даёт сдвиг логита $\varepsilon\|w\|_1$, суммирующий вклад всех пикселей.
- Градиент по входу — это $\pm w$, поэтому FGSM даёт картинку, повторяющую веса.